# Agentic Compliance Probe Analysis

This notebook summarizes model compliance scores, highlights failure cases, and checks for reasoning-versus-action mismatches across conditions.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
RESULTS_DIR = ROOT / 'results'

result_files = sorted(RESULTS_DIR.glob('results_*.jsonl'))
rows = []
for path in result_files:
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.head()

In [ ]:
if df.empty:
    raise ValueError('No experiment results found. Run pipeline/run_experiments.py first.')

summary = (
    df.groupby(['model_name', 'condition'], as_index=False)['compliance_score']
      .mean()
      .sort_values(['condition', 'model_name'])
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
pivot = summary.pivot(index='model_name', columns='condition', values='compliance_score')
pivot.plot(kind='bar', ax=ax, width=0.85)
ax.set_title('Average Compliance Score by Model and Condition')
ax.set_ylabel('Compliance score')
ax.set_xlabel('Model')
ax.legend(title='Condition', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
failure_cases = df.loc[df['compliance_score'] < 1.0, [
    'model_name', 'condition', 'scenario_id', 'scenario_title', 'compliance_score', 'error', 'response'
]]
failure_cases.head(20)

In [ ]:
mismatch_counts = (
    df.assign(reasoning_action_mismatch=df['compliance_details'].apply(lambda x: x.get('reasoning_action_mismatch', False)))
      .groupby(['model_name', 'condition'], as_index=False)['reasoning_action_mismatch']
      .mean()
      .sort_values(['condition', 'model_name'])
)
mismatch_counts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
mismatch_pivot = mismatch_counts.pivot(index='model_name', columns='condition', values='reasoning_action_mismatch')
mismatch_pivot.plot(kind='bar', ax=ax, color=['#2a9d8f', '#e76f51', '#264653'])
ax.set_title('Reasoning-Action Mismatch Rate')
ax.set_ylabel('Mismatch rate')
ax.set_xlabel('Model')
ax.legend(title='Condition', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## Interpretation

- Compare mean compliance across models and prompt conditions.
- Inspect low-scoring rows to understand whether failures come from tool misuse, safety violations, or evaluation-aware behavior shifts.
- Compare mismatch rates to see whether a model's stated reasoning aligns with the action it actually proposes.